# Memory-Safe COMET Evaluation

Optimized for MacBook Air M4: lazy audio loading, batch size 1, no `rows=list(ds)`, greedy decoding, unload baseline before loading fine-tuned model.

In [1]:
# Run once if COMET is missing.
# %pip install unbabel-comet sacrebleu


In [2]:
from pathlib import Path
import gc
import json

import numpy as np
import soundfile as sf
import torch
from tqdm.auto import tqdm
from transformers import WhisperForConditionalGeneration, WhisperProcessor


/Users/ihorivanyshyn/Documents/Audio_course/toronto/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
TEST_MANIFEST = Path('ua_ast_data/manifests/combined_uk/test.clean.jsonl')
BASE_MODEL = 'openai/whisper-base'
FINETUNED_MODEL = 'models/whisper-base-ua-ast'
PRED_DIR = Path('ua_ast_eval')
PRED_DIR.mkdir(parents=True, exist_ok=True)

# Set to None for FULL evaluation, or a number (e.g. 100) for a quick smoke test.
MAX_TEST_SAMPLES = None
BATCH_SIZE = 1
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print('Device:', DEVICE)

Device: mps


In [4]:
def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
        torch.mps.empty_cache()


def read_manifest(path: Path, max_samples: int | None = None) -> list[dict]:
    records = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            record = json.loads(line)
            if not record.get('target_text_uk', '').strip():
                continue
            if not Path(record['audio_path']).exists():
                continue
            records.append({
                'audio_path': record['audio_path'],
                'src': record['source_text'],
                'ref': record['target_text_uk'],
                'id': record['id'],
            })
            if max_samples is not None and len(records) >= max_samples:
                break
    return records

records = read_manifest(TEST_MANIFEST, MAX_TEST_SAMPLES)
print('Test samples:', len(records))

with (PRED_DIR / 'src.txt').open('w', encoding='utf-8') as f_src, (PRED_DIR / 'ref.txt').open('w', encoding='utf-8') as f_ref:
    for r in records:
        f_src.write(r['src'].replace(chr(10), ' ') + chr(10))
        f_ref.write(r['ref'].replace(chr(10), ' ') + chr(10))


Test samples: 100


In [5]:
def batched(items, batch_size):
    batch = []
    for item in items:
        batch.append(item)
        if len(batch) == batch_size:
            yield batch
            batch = []
    if batch:
        yield batch


def load_audio(path: str):
    audio, sr = sf.read(path, dtype='float32')
    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)
    if sr != 16000:
        raise ValueError(f'Expected 16 kHz audio, got {sr} for {path}')
    return audio


@torch.inference_mode()
def generate_hypotheses(model_name: str, out_path: Path) -> None:
    cleanup_memory()
    processor = WhisperProcessor.from_pretrained(model_name, language='Ukrainian', task='transcribe')
    model = WhisperForConditionalGeneration.from_pretrained(model_name).to(DEVICE)
    model.eval()
    model.config.use_cache = False
    forced_decoder_ids = processor.get_decoder_prompt_ids(language='Ukrainian', task='transcribe')

    done = 0
    if out_path.exists():
        done = sum(1 for _ in out_path.open(encoding='utf-8'))

    progress = tqdm(total=len(records), initial=done, desc=out_path.stem, unit='sample')
    remaining = records[done:]

    with out_path.open('a', encoding='utf-8') as f:
        for batch_records in batched(remaining, BATCH_SIZE):
            arrays = [load_audio(record['audio_path']) for record in batch_records]
            inputs = processor.feature_extractor(
                arrays,
                sampling_rate=16000,
                return_tensors='pt',
                padding='max_length',
                truncation=True,
                return_attention_mask=True,
            )
            input_features = inputs.input_features.to(DEVICE)
            predicted_ids = model.generate(
                input_features,
                forced_decoder_ids=forced_decoder_ids,
                max_new_tokens=160,
                num_beams=1,
                do_sample=False,
            )
            texts = processor.batch_decode(predicted_ids, skip_special_tokens=True)
            for text in texts:
                f.write(text.replace(chr(10), ' ').strip() + chr(10))
            f.flush()
            progress.update(len(batch_records))
            del arrays, inputs, input_features, predicted_ids
            cleanup_memory()

    progress.close()
    del model, processor
    cleanup_memory()


In [6]:
generate_hypotheses(BASE_MODEL, PRED_DIR / 'hyp_baseline.txt')

if Path(FINETUNED_MODEL).exists():
    generate_hypotheses(FINETUNED_MODEL, PRED_DIR / 'hyp_finetuned.txt')
else:
    print(f"Fine-tuned model {FINETUNED_MODEL} not found. Skipping its evaluation. You can train it later.")


hyp_finetuned: 100%|██████████| 100/100 [00:00<?, ?sample/s]


In [8]:
# COMET scoring. Requires: pip install unbabel-comet
from comet import download_model, load_from_checkpoint
import torch

model_path = download_model('Unbabel/wmt22-comet-da')
comet_model = load_from_checkpoint(model_path)

src_lines = (PRED_DIR / 'src.txt').read_text(encoding='utf-8').splitlines()
ref_lines = (PRED_DIR / 'ref.txt').read_text(encoding='utf-8').splitlines()

def score_file(hyp_path: Path):
    if not hyp_path.exists(): return None
    hyp_lines = hyp_path.read_text(encoding='utf-8').splitlines()
    data = [
        {'src': src, 'mt': hyp, 'ref': ref}
        for src, hyp, ref in zip(src_lines, hyp_lines, ref_lines)
    ]
    # Set num_workers >= 1 to prevent PyTorch ValueError when multiprocessing_context="fork" is set on MPS
    return comet_model.predict(
        data, 
        batch_size=2, 
        gpus=1 if torch.cuda.is_available() else 0,
        num_workers=2 if torch.backends.mps.is_available() else 0
    )

baseline = score_file(PRED_DIR / 'hyp_baseline.txt')
print('Baseline COMET:', baseline.system_score if baseline else "N/A")

finetuned = None
if Path(FINETUNED_MODEL).exists():
    finetuned = score_file(PRED_DIR / 'hyp_finetuned.txt')
    if finetuned:
        print('Fine-tuned COMET:', finetuned.system_score)
        print('Delta:', finetuned.system_score - baseline.system_score)

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 75983.77it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.1. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Encoder model frozen.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/Users/ihorivanyshyn/Documents/Audio_course/toronto/.venv/lib/python3.13/site-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try inst

Baseline COMET: 0.30741546511650086


/Users/ihorivanyshyn/Documents/Audio_course/toronto/.venv/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Predicting DataLoader 0: 100%|██████████| 50/50 [00:11<00:00,  4.50it/s]

Fine-tuned COMET: 0.36944365471601487
Delta: 0.062028189599514005


In [9]:
report = {
    'test_manifest': str(TEST_MANIFEST),
    'max_test_samples': MAX_TEST_SAMPLES,
    'baseline_model': BASE_MODEL,
    'baseline_comet': float(baseline.system_score) if baseline else None,
}

if finetuned:
    report['finetuned_model'] = FINETUNED_MODEL
    report['finetuned_comet'] = float(finetuned.system_score)
    report['delta_comet'] = float(finetuned.system_score - baseline.system_score)

(PRED_DIR / 'comet_report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
report

{'test_manifest': 'ua_ast_data/manifests/combined_uk/test.clean.jsonl',
 'max_test_samples': 100,
 'baseline_model': 'openai/whisper-base',
 'baseline_comet': 0.30741546511650086,
 'finetuned_model': 'models/whisper-base-ua-ast',
 'finetuned_comet': 0.36944365471601487,
 'delta_comet': 0.062028189599514005}

In [10]:
import random

def display_examples(num_examples=5):
    if not Path(PRED_DIR / 'hyp_finetuned.txt').exists():
        print("Fine-tuned hypothesis file not found. Run the evaluation cells first.")
        return
        
    hyp_base = (PRED_DIR / 'hyp_baseline.txt').read_text(encoding='utf-8').splitlines()
    hyp_fine = (PRED_DIR / 'hyp_finetuned.txt').read_text(encoding='utf-8').splitlines()
    
    # Using the 'records' list created earlier in the notebook
    num_total = len(records)
    indices = random.sample(range(num_total), min(num_examples, num_total))
    
    print(f"--- Showing {len(indices)} Random Evaluation Examples ---\n")
    for idx in indices:
        record = records[idx]
        print(f"Audio Path: {record['audio_path']}")
        print(f"English Source (src): {record['src']}")
        print(f"Target Ukrainian (ref): {record['ref']}")
        print(f"Baseline Whisper (hyp): {hyp_base[idx] if idx < len(hyp_base) else 'N/A'}")
        print(f"Fine-tuned Whisper (hyp): {hyp_fine[idx] if idx < len(hyp_fine) else 'N/A'}")
        print("-" * 50)

display_examples(5)

--- Showing 5 Random Evaluation Examples ---

Audio Path: ua_ast_data/audio/librispeech/test.clean/6930-75918-0019.wav
English Source (src): upon the large square in front of the hotel the shadows of the tents intersected by the golden moonbeams formed as it were a huge mosaic of jet and yellow flagstones
Target Ukrainian (ref): На великій площі перед готелем тіні шатрів, пересечених золотіми лунними променями, утворилися як величезний мозаїк джатів і жовтих флагманів.
Baseline Whisper (hyp): Апан, ларшу, в іншому відео, відео, шадоса, відео, іншого, зі голдин мумбім, зі відео, зі голдин мумбім, зі голдин мумбім, зі голдин мумбім, зі голдин мумбім, зі голдин мумбім, зі голдин мумбім, зі голдин мумбім, зі голдин мумбім, зі голдин мум
Fine-tuned Whisper (hyp): Подійшвидше, що в речі в іншому власній власній власній власній власній власній власній власній власній власній власній власній власній власній власній власній власній власній власній власній власній власній власній власній власній